In [1]:
# ============================================================
# Notebook    : 08_llm_narrative.ipynb
# Description : Automated risk narrative generation combining
#               rule-based baseline and LLM enrichment.
#               Pipeline:
#               Step 1 — Rule-based narrative (free, reproducible)
#               Step 2 — LLM enrichment via GPT-4o-mini
#               Step 3 — LLM-as-a-Judge quality evaluation
#               Input  : data/shap/07_idpol_shap_summary.csv
#               Output : data/narrative/08_narratives.csv
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install openai python-dotenv tqdm pandas numpy


# ============================================================
# 1. Libraries and environment setup
# ============================================================
import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm

os.makedirs("data/narrative", exist_ok=True)

# Load .env from the same folder as this notebook
load_dotenv(dotenv_path=os.path.join(os.path.dirname(
    os.path.abspath("__file__")), ".env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
LLM_MODEL      = os.getenv("LLM_MODEL", "gpt-4o-mini")

if not OPENAI_API_KEY:
    raise ValueError(
        "\n[ERROR] OPENAI_API_KEY is not set.\n"
        "  Add OPENAI_API_KEY=sk-... to your .env file."
    )

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"[CHECK 1] Model     : {LLM_MODEL}")
print(f"[CHECK 1] API key   : {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}")


# ============================================================
# 2. Load IDpol-level SHAP summary (from notebook 07)
# ============================================================
df = pd.read_csv("data/shap/07_idpol_shap_summary.csv")
grade_summary = pd.read_csv("data/shap/07_grade_summary.csv")
feat_imp      = pd.read_csv("data/shap/07_feature_importance.csv",
                             index_col=0)

print(f"\n[CHECK 2] SHAP summary shape: {df.shape}")
print(f"[CHECK 2] Columns: {list(df.columns)}")
print(f"[CHECK 2] Risk grade counts:\n"
      f"{df['risk_grade'].value_counts().sort_index()}")


# ============================================================
# 3. Feature metadata for narrative generation
#    - Human-readable descriptions for each feature and
#      its SHAP direction (positive = increases risk)
# ============================================================
FEATURE_META = {
    "VehPower" : {
        "label"   : "Vehicle Power",
        "high_risk": "high-power vehicle (increases risk)",
        "low_risk" : "low-power vehicle (reduces risk)",
    },
    "VehType"  : {
        "label"   : "Vehicle Type",
        "high_risk": "high-risk vehicle type",
        "low_risk" : "low-risk vehicle type",
    },
    "Usage"    : {
        "label"   : "Usage Pattern",
        "high_risk": "high-intensity usage pattern",
        "low_risk" : "low-intensity usage pattern",
    },
    "Expo"     : {
        "label"   : "Exposure (annual fraction)",
        "high_risk": "high annual exposure",
        "low_risk" : "low annual exposure",
    },
    "YearGap"  : {
        "label"   : "Observation Year Gap",
        "high_risk": "late observation period",
        "low_risk" : "early observation period",
    },
}

SHAP_COLS = ["Expo", "YearGap", "Usage", "VehType", "VehPower"]

print(f"\n[CHECK 3] Feature metadata loaded for: {list(FEATURE_META.keys())}")


# ============================================================
# 4. Step 1 — Rule-based narrative generation (baseline)
#    - Deterministic, reproducible, zero cost
#    - Identifies top contributing features and their direction
#    - Assigns a risk sentence per grade
# ============================================================
GRADE_SENTENCES = {
    "G1_Safe"    : "This policyholder is in the SAFE tier. "
                   "Risk indicators are below average across all features.",
    "G2_Caution" : "This policyholder is in the CAUTION tier. "
                   "Mild risk signals are present; routine monitoring recommended.",
    "G3_Risk"    : "This policyholder is in the RISK tier. "
                   "Multiple risk factors are elevated; closer review is warranted.",
    "G4_HighRisk": "This policyholder is in the HIGH-RISK tier. "
                   "Significant risk contributors identified; premium adjustment advised.",
    "G5_Critical": "This policyholder is in the CRITICAL tier. "
                   "Severe risk profile; immediate underwriting intervention recommended.",
}

def build_rule_narrative(row):
    """Generate deterministic rule-based narrative from SHAP values."""
    grade   = row["risk_grade"]
    shap_vals = {col: row[col] for col in SHAP_COLS}

    # Sort features by absolute SHAP contribution
    sorted_feats = sorted(shap_vals.items(),
                          key=lambda x: abs(x[1]), reverse=True)

    # Top 2 risk-increasing features
    risk_drivers = [(f, v) for f, v in sorted_feats if v > 0][:2]
    # Top 1 risk-reducing feature
    protect_feat = [(f, v) for f, v in sorted_feats if v < 0][:1]

    grade_sent = GRADE_SENTENCES.get(grade, "Risk grade undetermined.")

    driver_parts = []
    for feat, val in risk_drivers:
        meta = FEATURE_META.get(feat, {"label": feat,
                                        "high_risk": "elevated"})
        driver_parts.append(
            f"{meta['label']} ({meta['high_risk']}, SHAP={val:+.3f})"
        )

    protect_parts = []
    for feat, val in protect_feat:
        meta = FEATURE_META.get(feat, {"label": feat,
                                        "low_risk": "reduced"})
        protect_parts.append(
            f"{meta['label']} ({meta['low_risk']}, SHAP={val:+.3f})"
        )

    narrative = grade_sent
    if driver_parts:
        narrative += (f" Primary risk contributors: "
                      f"{'; '.join(driver_parts)}.")
    if protect_parts:
        narrative += (f" Mitigating factor: "
                      f"{'; '.join(protect_parts)}.")
    narrative += (f" Predicted claim probability: "
                  f"{row['Prob']:.1%} "
                  f"(observed over {int(row['n_years'])} year(s), "
                  f"{int(row['year_min'])}–{int(row['year_max'])}).")
    return narrative

print(f"\n[CHECK 4] Generating rule-based narratives...")
df["narrative_rule"] = df.apply(build_rule_narrative, axis=1)
print(f"[CHECK 4] Sample rule-based narrative (first row):")
print(f"  {df['narrative_rule'].iloc[0]}")


# ============================================================
# 5. Step 2 — LLM enrichment via GPT-4o-mini
#    - Input  : rule-based narrative + raw SHAP values
#    - Output : enriched, professional underwriting narrative
#    - Batch  : process N_SAMPLE policyholders for cost control
#      (full run: set N_SAMPLE = len(df))
# ============================================================
N_SAMPLE   = 100    # set to len(df) for full run; 100 for cost-controlled demo
SLEEP_SEC  = 0.5    # rate-limit buffer between API calls

SYSTEM_PROMPT = """You are an expert insurance underwriting analyst.
Your task is to enrich a rule-based risk summary into a concise,
professional underwriting narrative in English.

Requirements:
- Length: 3-4 sentences maximum
- Tone: professional, objective, actionable
- Include: risk grade, top risk driver, claim probability, recommendation
- Do NOT invent facts not present in the input
- Do NOT use bullet points — write flowing prose only"""

def build_user_prompt(row):
    shap_str = ", ".join(
        f"{col}={row[col]:+.4f}" for col in SHAP_COLS
    )
    return (
        f"Rule-based summary: {row['narrative_rule']}\n"
        f"SHAP values: {shap_str}\n"
        f"Risk grade: {row['risk_grade']}\n"
        f"Predicted claim probability: {row['Prob']:.4f}\n"
        f"Observation years: {int(row['year_min'])}–{int(row['year_max'])}\n\n"
        f"Please enrich this into a professional underwriting narrative."
    )

def call_llm(row, retries=3):
    """Call GPT-4o-mini with retry logic."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": build_user_prompt(row)},
                ],
                temperature=0.3,
                max_tokens=256,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                return f"[ERROR] {str(e)}"

sample_df = df.head(N_SAMPLE).copy()

print(f"\n[CHECK 5] Calling GPT-4o-mini for {N_SAMPLE} policyholders...")
print(f"[CHECK 5] Estimated cost: ~${N_SAMPLE * 0.0002:.2f} USD "
      f"(gpt-4o-mini ~$0.0002/call at 256 tokens)")

llm_narratives = []
for _, row in tqdm(sample_df.iterrows(),
                   total=len(sample_df),
                   desc="LLM narrative"):
    narrative = call_llm(row)
    llm_narratives.append(narrative)
    time.sleep(SLEEP_SEC)

sample_df["narrative_llm"] = llm_narratives

print(f"\n[CHECK 5] Sample LLM narrative (first row):")
print(f"  {sample_df['narrative_llm'].iloc[0]}")


# ============================================================
# 6. Step 3 — LLM-as-a-Judge quality evaluation
#    - Each LLM narrative is evaluated by the same model
#      on three dimensions (proposal Section 3-4):
#      (a) Accuracy   : factual consistency with SHAP input
#      (b) Utility    : actionability for underwriters
#      (c) Fairness   : absence of discriminatory language
#    - Score: 1-5 per dimension, JSON output
# ============================================================
JUDGE_SYSTEM = """You are an expert evaluator of insurance underwriting narratives.
Evaluate the given narrative on three dimensions and return ONLY a JSON object.

Scoring criteria (1=poor, 5=excellent):
- accuracy  : Is the narrative factually consistent with the SHAP input provided?
- utility   : Is the narrative actionable and useful for an underwriter?
- fairness  : Is the narrative free from discriminatory or biased language?

Return format (JSON only, no extra text):
{"accuracy": <int>, "utility": <int>, "fairness": <int>, "comment": "<one sentence>"}"""

def judge_narrative(row):
    """Evaluate LLM narrative quality."""
    user_msg = (
        f"SHAP input: {build_user_prompt(row)}\n\n"
        f"Narrative to evaluate:\n{row['narrative_llm']}"
    )
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.0,
                max_tokens=128,
            )
            raw = response.choices[0].message.content.strip()
            # robust JSON extraction
            start = raw.find("{")
            end   = raw.rfind("}") + 1
            return json.loads(raw[start:end])
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                return {"accuracy": 0, "utility": 0,
                        "fairness": 0, "comment": f"ERROR: {e}"}

print(f"\n[CHECK 6] Running LLM-as-a-Judge evaluation...")
judge_results = []
for _, row in tqdm(sample_df.iterrows(),
                   total=len(sample_df),
                   desc="LLM-as-Judge"):
    result = judge_narrative(row)
    judge_results.append(result)
    time.sleep(SLEEP_SEC)

judge_df = pd.DataFrame(judge_results)
sample_df["judge_accuracy"] = judge_df["accuracy"].values
sample_df["judge_utility"]  = judge_df["utility"].values
sample_df["judge_fairness"] = judge_df["fairness"].values
sample_df["judge_comment"]  = judge_df["comment"].values

print(f"\n[CHECK 6] LLM-as-a-Judge scores (mean over {N_SAMPLE} samples):")
for dim in ["accuracy", "utility", "fairness"]:
    col  = f"judge_{dim}"
    mean = sample_df[col].replace(0, np.nan).mean()
    std  = sample_df[col].replace(0, np.nan).std()
    print(f"  {dim:12s}: {mean:.2f} ± {std:.2f} (out of 5)")


# ============================================================
# 7. Comparison — rule-based vs LLM narrative (length & quality)
# ============================================================
sample_df["rule_len"] = sample_df["narrative_rule"].str.split().str.len()
sample_df["llm_len"]  = sample_df["narrative_llm"].str.split().str.len()

print(f"\n[CHECK 7] Narrative length comparison:")
print(f"  Rule-based : {sample_df['rule_len'].mean():.1f} words (mean)")
print(f"  LLM        : {sample_df['llm_len'].mean():.1f} words (mean)")

# Quality score by risk grade
grade_quality = sample_df.groupby("risk_grade")[
    ["judge_accuracy", "judge_utility", "judge_fairness"]
].mean().round(2)
print(f"\n[CHECK 7] LLM-as-a-Judge scores by risk grade:")
print(grade_quality.to_string())


# ============================================================
# 8. Save outputs
# ============================================================

# Full rule-based narratives (all policyholders)
df_full_out = df[["IDpol", "risk_grade", "Prob", "Label",
                   "n_years", "year_min", "year_max",
                   "narrative_rule"] + SHAP_COLS].copy()
df_full_out.to_csv("data/narrative/08_rule_narratives_all.csv",
                   index=False, encoding="utf-8-sig")

# LLM-enriched narratives + judge scores (sample)
sample_out = sample_df[["IDpol", "risk_grade", "Prob", "Label",
                          "narrative_rule", "narrative_llm",
                          "judge_accuracy", "judge_utility",
                          "judge_fairness", "judge_comment"]].copy()
sample_out.to_csv("data/narrative/08_llm_narratives_sample.csv",
                  index=False, encoding="utf-8-sig")

# Judge summary table (for paper)
judge_summary = pd.DataFrame({
    "dimension": ["accuracy", "utility", "fairness"],
    "mean_score": [
        sample_df["judge_accuracy"].replace(0, np.nan).mean(),
        sample_df["judge_utility"].replace(0, np.nan).mean(),
        sample_df["judge_fairness"].replace(0, np.nan).mean(),
    ],
    "std_score": [
        sample_df["judge_accuracy"].replace(0, np.nan).std(),
        sample_df["judge_utility"].replace(0, np.nan).std(),
        sample_df["judge_fairness"].replace(0, np.nan).std(),
    ],
}).round(3)
judge_summary.to_csv("data/narrative/08_judge_summary.csv",
                     index=False, encoding="utf-8-sig")

print(f"\n[CHECK 8] Saved files:")
print(f"  data/narrative/08_rule_narratives_all.csv  "
      f"({len(df_full_out):,} policyholders)")
print(f"  data/narrative/08_llm_narratives_sample.csv  "
      f"({len(sample_out):,} policyholders)")
print(f"  data/narrative/08_judge_summary.csv")


# ============================================================
# 9. Sample narrative showcase (for paper appendix)
# ============================================================
print(f"\n===== Sample Narrative Showcase =====")
for grade in ["G1_Safe", "G3_Risk", "G5_Critical"]:
    row = sample_df[sample_df["risk_grade"] == grade].iloc[0]
    print(f"\n[{grade}] IDpol: {row['IDpol']}, "
          f"Prob: {row['Prob']:.3f}")
    print(f"  RULE : {row['narrative_rule']}")
    print(f"  LLM  : {row['narrative_llm']}")
    print(f"  JUDGE: accuracy={row['judge_accuracy']}, "
          f"utility={row['judge_utility']}, "
          f"fairness={row['judge_fairness']}")
    print(f"  NOTE : {row['judge_comment']}")


# ============================================================
# 10. Summary
# ============================================================
print(f"\n===== Notebook 08 Summary =====")
print(f"Rule-based narratives  : {len(df_full_out):,} policyholders (all)")
print(f"LLM narratives         : {N_SAMPLE} policyholders (cost-controlled sample)")
print(f"LLM model              : {LLM_MODEL}")
print(f"LLM-as-a-Judge scores  :")
for _, row in judge_summary.iterrows():
    print(f"  {row['dimension']:12s}: {row['mean_score']:.2f} ± "
          f"{row['std_score']:.2f} / 5.0")
print(f"Outputs saved to       : data/narrative/")
print(f"================================")

[CHECK 1] Model     : gpt-4o-mini
[CHECK 1] API key   : sk-proj-...sJxU

[CHECK 2] SHAP summary shape: (10709, 14)
[CHECK 2] Columns: ['IDpol', 'Expo', 'YearGap', 'Usage', 'VehType', 'VehPower', 'mean_risk_shap', 'Label', 'Prob', 'n_years', 'year_min', 'year_max', 'risk_grade', 'cohort']
[CHECK 2] Risk grade counts:
risk_grade
G1_Safe        2142
G2_Caution     2142
G3_Risk        2141
G4_HighRisk    2142
G5_Critical    2142
Name: count, dtype: int64

[CHECK 3] Feature metadata loaded for: ['VehPower', 'VehType', 'Usage', 'Expo', 'YearGap']

[CHECK 4] Generating rule-based narratives...
[CHECK 4] Sample rule-based narrative (first row):
  This policyholder is in the SAFE tier. Risk indicators are below average across all features. Primary risk contributors: Observation Year Gap (late observation period, SHAP=+0.076); Exposure (annual fraction) (high annual exposure, SHAP=+0.054). Mitigating factor: Vehicle Type (low-risk vehicle type, SHAP=-1.017). Predicted claim probability: 4.2% (ob

LLM narrative: 100%|█████████████████████████████████████████████████████████████████| 100/100 [04:48<00:00,  2.89s/it]



[CHECK 5] Sample LLM narrative (first row):
  The policyholder is classified within the SAFE tier, receiving a risk grade of G1_Safe, indicating a favorable risk profile. Key risk drivers include a late observation period and high annual exposure, contributing to a predicted claim probability of 4.2% based on historical data from 2004 to 2007. However, the low-risk vehicle type serves as a significant mitigating factor, enhancing the overall risk assessment. Given these insights, it is recommended to proceed with the policy, acknowledging the manageable risk level.

[CHECK 6] Running LLM-as-a-Judge evaluation...


LLM-as-Judge: 100%|██████████████████████████████████████████████████████████████████| 100/100 [02:54<00:00,  1.75s/it]


[CHECK 6] LLM-as-a-Judge scores (mean over 100 samples):
  accuracy    : 5.00 ± 0.00 (out of 5)
  utility     : 5.00 ± 0.00 (out of 5)
  fairness    : 5.00 ± 0.00 (out of 5)

[CHECK 7] Narrative length comparison:
  Rule-based : 46.4 words (mean)
  LLM        : 84.4 words (mean)

[CHECK 7] LLM-as-a-Judge scores by risk grade:
             judge_accuracy  judge_utility  judge_fairness
risk_grade                                                
G1_Safe                 5.0            5.0             5.0
G2_Caution              5.0            5.0             5.0
G3_Risk                 5.0            5.0             5.0
G4_HighRisk             5.0            5.0             5.0
G5_Critical             5.0            5.0             5.0

[CHECK 8] Saved files:
  data/narrative/08_rule_narratives_all.csv  (10,709 policyholders)
  data/narrative/08_llm_narratives_sample.csv  (100 policyholders)
  data/narrative/08_judge_summary.csv

===== Sample Narrative Showcase =====

[G1_Safe] IDpol: PN10